### Using two tables/stores  

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from db import init_db, create_conversation, save_message, load_history

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the context below. If it's not there, say you don't know.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

In [ ]:
chain = prompt | llm

In [ ]:
def ask(conversation_id: str, question: str) -> str:
    # 1. load previous messages (before saving the new one)
    history = [
        HumanMessage(c) if r == "user" else AIMessage(c)
        for r, c in load_history(conversation_id)
    ]

    # 2. retrieve from pgvector
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)

    # 3. generate
    answer = chain.invoke(
        {"context": context, "history": history, "question": question}
    ).content

    # 4. persist both turns
    save_message(conversation_id, "user", question)
    save_message(
        conversation_id, "assistant", answer,
        sources=[d.metadata for d in docs],   # must be JSON-serializable
    )
    return answer

In [ ]:
if __name__ == "__main__":
    init_db()                       # once is enough, harmless to repeat
    conv_id = create_conversation(user_id="tanmay")
    print(ask(conv_id, "What is this document about?"))
    print(ask(conv_id, "Summarize that in one line."))   # uses stored history